In [31]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [32]:
words=open('names.txt', 'r').read().splitlines()
print(words[:8:])
print(len(words))

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
32033


In [33]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [34]:
#how to data is being split and input
block_size=3  #no. of elements in our context used to predict the next char 
X,Y=[],[]
for w in words[:5]:
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i]for i in context),'--->',ch)
        context=context[1:]+[ix] #updating last (3rd) element in context block

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [35]:
X=torch.tensor(X)
Y =torch.tensor(Y)
print(X.shape, X.dtype, Y.shape, Y.dtype)

torch.Size([32, 3]) torch.int64 torch.Size([32]) torch.int64


In [ ]:
g=torch.Generator().manual_seed(2147483647)

#initializing 2-d embeddings for all 27 of our characters
C=torch.randn((27,2),generator=g)

#creating hyperparamters
W1=torch.randn((6,100),generator=g)
b1=torch.randn((100),generator=g)

W2=torch.randn((100,27),generator=g)
b2=torch.randn((27),generator=g)

In [51]:
emb=C[X]  #extracting embeddings of samples stored in X
print("emb shape:",emb.shape)
h=torch.tanh(emb.view(-1,6)@ W1 + b1 ) #calculating 2nd layer
print(h.shape,"\n",h)

emb shape: torch.Size([32, 3, 2])
torch.Size([32, 100]) 
 tensor([[-0.9348,  1.0000,  0.9258,  ...,  0.9786, -0.1926,  0.9515],
        [ 0.2797,  0.9997,  0.7675,  ...,  0.9929,  0.9992,  0.9981],
        [-0.9960,  1.0000, -0.8694,  ..., -0.5159, -1.0000, -0.0069],
        ...,
        [-0.9996,  1.0000, -0.9273,  ..., -0.9999, -0.9974, -0.9970],
        [-0.9043,  1.0000,  0.9868,  ..., -0.7859, -0.4819,  0.9981],
        [-0.9048,  1.0000,  0.9553,  ...,  0.9866,  1.0000,  0.9907]])


In [52]:
logits=h @ W2 +b2      #calculating 3rd layer
print("logits shape:",logits.shape)

logits shape: torch.Size([32, 27])


In [53]:
loss=F.cross_entropy(logits,Y)  #Y contains all the actual labels of our samples in X
loss

tensor(17.7697)